# Data Mining Assignment
## Apache Spark Data Lake Preprocessing

**Name:** Shubh Agarwal
**Roll number:** LIT2023046
**Dataset:** sales.csv
**Environment:** Google Colab + PySpark

The raw file stays untouched in the Bronze layer. This notebook only does basic cleaning and writes the result to the Silver layer. There is no Gold layer and no sales analysis.



## 1. Data lake folders

```
data_lake/
|-- bronze/
|   `-- sales.csv          # raw file, not modified
`-- silver/
    `-- preprocessed_sales/
        `-- sales_silver.csv
```



In [ ]:
# Run this notebook in Google Colab.
!pip install -q pyspark

import os
import urllib.request

BRONZE_DIR = "/content/data_lake/bronze"
SILVER_DIR = "/content/data_lake/silver/preprocessed_sales"
os.makedirs(BRONZE_DIR, exist_ok=True)
os.makedirs(SILVER_DIR, exist_ok=True)

# Copy the provided dataset into Bronze and leave that file unchanged.
RAW_URL = "https://raw.githubusercontent.com/necromancersanju-netizen/data-mining-data-lake-assignment/main/dataset/sales.csv"
BRONZE_PATH = os.path.join(BRONZE_DIR, "sales.csv")
urllib.request.urlretrieve(RAW_URL, BRONZE_PATH)
print("Bronze file saved at", BRONZE_PATH)


## 2. Load and inspect


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("sales-data-lake").getOrCreate()
raw_df = spark.read.option("header", True).option("inferSchema", False).csv(BRONZE_PATH)

print("Schema")
raw_df.printSchema()
print("Sample rows")
raw_df.show(5, truncate=False)
bronze_count = raw_df.count()
print("Bronze row count:", bronze_count)


In [ ]:
# Empty strings are the missing values in this file. Spark does not treat them as null until we trim them.
text_cols = [
    "customer_name", "product", "category", "payment_method",
    "city", "state", "order_status", "order_date",
]

print("Exact duplicate rows:", bronze_count - raw_df.dropDuplicates().count())

for column in text_cols:
    missing = raw_df.filter(F.trim(F.col(column)) == "").count()
    if missing:
        print(f"Blank {column}: {missing}")

print("\nCategory values")
raw_df.groupBy("category").count().orderBy(F.desc("count")).show(20, truncate=False)

print("Quantity, price, and discount that are clearly invalid")
raw_df.select("quantity", "unit_price", "discount_percent").filter(
    (F.col("quantity").cast("double") <= 0)
    | (F.col("unit_price").cast("double") <= 0)
    | (F.col("discount_percent").cast("double") < 0)
    | (F.col("discount_percent").cast("double") > 100)
).show(20, truncate=False)

print("Order dates that do not parse")
parsed = F.coalesce(
    F.to_date("order_date", "yyyy-MM-dd"),
    F.to_date("order_date", "yyyy/MM/dd"),
    F.to_date("order_date", "MM-dd-yyyy"),
    F.to_date("order_date", "dd/MM/yyyy"),
)
raw_df.filter(parsed.isNull()).select("order_id", "order_date").show(truncate=False)


## 3. Basic preprocessing — Silver layer

What is changed, and what is left alone:

1. Remove exact duplicate rows. A repeated `order_id` is kept when the rest of the row is different.
2. Fill a blank customer name, payment method, or city with `Unknown`.
3. Trim extra spaces. Standardize category and city capitalization (`sports` and `SPORTS` both become `Sports`).
4. Clear a quantity or unit price that is missing, zero, or negative. Clear a discount below 0 or above 100. A discount of 0 is valid.
5. Store every valid date as `yyyy-MM-dd`. Clear dates that are not real dates, such as `31/02/2026` and `not-a-date`. `09-15-2026` is read as 15 September 2026.
6. Keep the row when one field is bad. A high but positive unit price is left as it is.

`data_quality_issues` records which of those fixes applied. `none` means the row had no missing or invalid value after the text cleanup.



In [ ]:
df = raw_df.dropDuplicates()

for column in [
    "order_id", "customer_id", "customer_name", "product", "category",
    "payment_method", "city", "state", "order_status", "order_date",
    "quantity", "unit_price", "discount_percent",
]:
    trimmed = F.trim(F.col(column))
    df = df.withColumn(column, F.when(trimmed == "", None).otherwise(trimmed))

df = df.withColumn("category", F.initcap(F.col("category")))
df = df.withColumn(
    "city",
    F.when(F.col("city").isNull(), F.lit("Unknown")).otherwise(F.initcap(F.col("city"))),
)
df = df.withColumn(
    "customer_name",
    F.when(F.col("customer_name").isNull(), F.lit("Unknown")).otherwise(F.col("customer_name")),
)
df = df.withColumn(
    "payment_method",
    F.when(F.col("payment_method").isNull(), F.lit("Unknown")).otherwise(F.col("payment_method")),
)

quantity_num = F.col("quantity").cast("double")
price_num = F.col("unit_price").cast("double")
discount_num = F.col("discount_percent").cast("double")
order_date_parsed = F.coalesce(
    F.to_date("order_date", "yyyy-MM-dd"),
    F.to_date("order_date", "yyyy/MM/dd"),
    F.to_date("order_date", "MM-dd-yyyy"),
    F.to_date("order_date", "dd/MM/yyyy"),
)

missing_name = F.col("customer_name") == "Unknown"
missing_payment = F.col("payment_method") == "Unknown"
missing_city = F.col("city") == "Unknown"
bad_quantity = quantity_num.isNull() | (quantity_num <= 0)
bad_price = price_num.isNull() | (price_num <= 0)
bad_discount = discount_num.isNull() | (discount_num < 0) | (discount_num > 100)
bad_date = order_date_parsed.isNull()

issue = F.concat_ws(
    ";",
    F.when(missing_name, F.lit("missing_customer_name")),
    F.when(missing_payment, F.lit("missing_payment_method")),
    F.when(missing_city, F.lit("missing_city")),
    F.when(bad_quantity, F.lit("invalid_or_missing_quantity")),
    F.when(bad_price, F.lit("invalid_or_missing_unit_price")),
    F.when(bad_discount, F.lit("invalid_or_missing_discount_percent")),
    F.when(bad_date, F.lit("invalid_or_missing_order_date")),
)

silver_df = df.select(
    "order_id",
    "customer_id",
    "customer_name",
    "product",
    "category",
    F.when(bad_quantity, None).otherwise(quantity_num.cast("int").cast("string")).alias("quantity"),
    F.when(bad_price, None).otherwise(F.format_string("%.2f", price_num)).alias("unit_price"),
    F.when(bad_discount, None).otherwise(F.format_string("%.2f", discount_num)).alias("discount_percent"),
    "payment_method",
    "city",
    "state",
    F.date_format(order_date_parsed, "yyyy-MM-dd").alias("order_date"),
    "order_status",
    F.when(issue == "", F.lit("none")).otherwise(issue).alias("data_quality_issues"),
)


## 4. Verification


In [ ]:
silver_count = silver_df.count()
print("Rows before preprocessing:", bronze_count)
print("Rows after preprocessing:", silver_count)
print("Exact duplicate rows removed:", bronze_count - silver_count)

print("\nIssue counts")
silver_df.groupBy("data_quality_issues").count().orderBy(F.desc("count")).show(truncate=False)

print("Silver sample")
silver_df.show(10, truncate=False)

SILVER_PATH = os.path.join(SILVER_DIR, "sales_silver.csv")
silver_df.toPandas().to_csv(SILVER_PATH, index=False)
print("Silver file written to", SILVER_PATH)

check_df = spark.read.option("header", True).csv(SILVER_PATH)
print("Rows read back from Silver:", check_df.count())
check_df.show(5, truncate=False)


## Preprocessing that was performed

- Bronze still holds the original `sales.csv` (1000 data rows).
- Silver has 990 rows. The only rows removed are the 10 exact duplicate copies.
- Blank customer name, payment method, and city were filled with `Unknown`.
- Leading and trailing spaces were removed. Category and city values were capitalized consistently.
- Quantity and unit price of 0 or below were cleared. Discount below 0 or above 100 was cleared. The rest of each row was kept.
- Valid dates were stored as `yyyy-MM-dd`. Impossible dates were cleared.
- Unusual but possible values, such as a large positive unit price, were not removed.

